# 01 — Préparation et exploration des données
## Classification des intentions bancaires — BANKING77
**Objectif du projet :** prédire l'intention d'un message bancaire parmi 77 catégories, puis comparer une approche TF-IDF + régression logistique à DistilBERT fine-tuné.

Ce notebook charge les CSV, examine les données, reprend ton nettoyage et crée un découpage reproductible (80 % entraînement, 20 % validation).

**Entrées :** `banking_data/train.csv` et `banking_data/test.csv`, avec les colonnes `text` et `category`.
**Sorties :** les données préparées et l'ordre des catégories dans `data/processed/`.

Exécuter les notebooks dans l'ordre **01 → 02 → 03**, chacun de haut en bas. Aucun état en mémoire n'est partagé entre eux.

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Ouvrir Jupyter depuis le dossier projet_banking77.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"
for directory in (DATA_DIR, RESULTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

from sklearn.model_selection import train_test_split

## 1. Charger les données

In [ ]:
path1 = PROJECT_DIR / "banking_data" / "train.csv"
path2 = PROJECT_DIR / "banking_data" / "test.csv"
for path in (path1, path2):
    if not path.exists():
        raise FileNotFoundError(f"Ajouter le fichier de données : {path}")
train_data = pd.read_csv(path1)
test_data = pd.read_csv(path2)
for name, frame in (("train", train_data), ("test", test_data)):
    assert {"text", "category"}.issubset(frame.columns), f"Colonnes text/category absentes du {name}."
print("Nombre de messages :", len(train_data))
print("Nombre de catégories du train :", train_data["category"].nunique())
train_data.head()

## 2. Nettoyer les données
La fonction `text_key` sert à repérer les doublons ; les textes transmis aux modèles restent ceux d'origine. Ton nettoyage retire les lignes invalides, les textes aux étiquettes contradictoires, les doublons et les textes également présents dans le test.

Le test est conservé pour l'évaluation finale. Ses textes servent uniquement au contrôle des recouvrements, comme dans ton code initial.

In [ ]:
def text_key(series):
    return (series.fillna("").astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip())

def invalid_rows(frame):
    return (frame[["text", "category"]].isna().any(axis=1) | text_key(frame["text"]).eq("") | text_key(frame["category"]).eq(""))

In [ ]:
assert not invalid_rows(test_data).any(), "Le test contient des lignes invalides à examiner."

clean = train_data.loc[~invalid_rows(train_data)].copy()
clean["source_row"] = clean.index
clean["text_key"] = text_key(clean["text"])

label_counts = clean.groupby("text_key")["category"].nunique()
conflicts = label_counts[label_counts > 1].index
print("Textes associés à plusieurs catégories :", len(conflicts))
clean = clean.loc[~clean["text_key"].isin(conflicts)].copy()
clean = clean.drop_duplicates(subset="text_key").copy()

test_keys = set(text_key(test_data["text"]))
overlap = clean["text_key"].isin(test_keys)
print("Textes du train présents dans le test :", int(overlap.sum()))
clean = clean.loc[~overlap].reset_index(drop=True)

print("Train après nettoyage :", len(clean))
print("Lignes retirées au total :", len(train_data) - len(clean))
assert clean["category"].nunique() == 77
assert clean["text_key"].is_unique

## 3. Explorer les catégories et la longueur des messages

In [ ]:
counts = clean["category"].value_counts()
print("Nombre d'exemples par catégorie :")
print(counts.describe().round(2).to_string())
print("\nLes 5 catégories les moins représentées :")
print(counts.tail(5).to_string())
word_lengths = clean["text"].str.split().str.len()
print("\nLongueur des messages en mots (pas en tokens DistilBERT) :")
print(word_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.sort_values().head(10).plot.barh(ax=axes[0], title="10 catégories les moins représentées")
axes[0].set_xlabel("Nombre de messages")
word_lengths.plot.hist(bins=30, ax=axes[1], title="Longueur des messages")
axes[1].set_xlabel("Nombre de mots")
axes[1].set_ylabel("Nombre de messages")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "exploration.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Séparer entraînement et validation
La stratification conserve la répartition des catégories. La graine 42 rend le découpage reproductible.

In [ ]:
train_df, val_df = train_test_split(clean, test_size=0.20, random_state= 42, stratify=clean["category"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [ ]:
train_df["labels"], categories = pd.factorize(train_df["category"])
val_df["labels"] = categories.get_indexer(val_df["category"])
assert (val_df["labels"] >= 0).all()
assert set(train_df["text_key"]).isdisjoint(val_df["text_key"])
assert set(clean["text_key"]).isdisjoint(test_keys)
print(f"Train : {len(train_df)} | Validation : {len(val_df)} | Test : {len(test_data)}")

## 5. Sauvegarder les données pour les deux modèles

In [ ]:
train_df.to_csv(DATA_DIR / "train.csv", index=False)
val_df.to_csv(DATA_DIR / "validation.csv", index=False)
test_data.to_csv(DATA_DIR / "test.csv", index=False)
(DATA_DIR / "categories.json").write_text(
    json.dumps(categories.tolist(), ensure_ascii=False, indent=2), encoding="utf-8"
)
summary = {
    "train_initial": len(train_data), "train_nettoye": len(clean),
    "train": len(train_df), "validation": len(val_df), "test": len(test_data),
    "categories": len(categories), "random_state": 42,
}
(RESULTS_DIR / "data_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Données sauvegardées. Passer au notebook 02.")